IHK

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('Merged_IHK_Februari.csv')

# Melihat 5 data awal
print(df.head())

In [ ]:
print(df.columns)

In [ ]:
long_df = df.melt(
    id_vars='Daerah',
    var_name='Tahun',
    value_name='IHK'
)

long_df['IHK'] = pd.to_numeric(
    long_df['IHK'],
    errors='coerce'
)

In [ ]:
long_df['Tahun'] = long_df['Tahun'].astype(int)

In [ ]:
long_df = long_df.sort_values(
    ['Daerah', 'Tahun']
)

In [ ]:
print(long_df.isnull().sum())

In [ ]:
long_df['IHK'] = long_df['IHK'].fillna(
    long_df['IHK'].median()
)

In [ ]:
print(long_df.info())

In [ ]:
sample = long_df[
    long_df['Daerah'] == long_df['Daerah'].iloc[0]
]

plt.figure(figsize=(10,5))

plt.plot(
    sample['Tahun'],
    sample['IHK'],
    marker='o'
)

plt.title("Tren IHK")

plt.xlabel("Tahun")
plt.ylabel("IHK")

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(data=long_df['IHK'])

plt.title("Boxplot IHK")

plt.show()

In [ ]:
clean_df = long_df.copy()

lower = clean_df['IHK'].quantile(0.10)
upper = clean_df['IHK'].quantile(0.90)

clean_df['IHK'] = np.clip(
    clean_df['IHK'],
    lower,
    upper
)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,5))

sns.boxplot(
    data=long_df['IHK'],
    ax=axes[0]
)

axes[0].set_title("Sebelum Winsorization")

sns.boxplot(
    data=clean_df['IHK'],
    ax=axes[1]
)

axes[1].set_title("Sesudah Winsorization")

plt.show()

In [ ]:
clean_df['Trend'] = (
    clean_df['Tahun']
    - clean_df['Tahun'].min()
)

In [ ]:
clean_df['Lag1'] = (
    clean_df.groupby('Daerah')['IHK']
    .shift(1)
)

In [ ]:
clean_df['Lag2'] = (
    clean_df.groupby('Daerah')['IHK']
    .shift(2)
)

In [ ]:
clean_df['Lag3'] = (
    clean_df.groupby('Daerah')['IHK']
    .shift(3)
)

In [ ]:
clean_df['Diff1'] = (
    clean_df.groupby('Daerah')['IHK']
    .diff(1)
)

In [ ]:
clean_df['Growth_%'] = (
    clean_df.groupby('Daerah')['IHK']
    .pct_change()
) * 100

In [ ]:
clean_df['RollingMean3'] = (

    clean_df.groupby('Daerah')['IHK']

    .rolling(3)

    .mean()

    .reset_index(0, drop=True)
)

In [ ]:
clean_df['RollingSTD3'] = (

    clean_df.groupby('Daerah')['IHK']

    .rolling(3)

    .std()

    .reset_index(0, drop=True)
)

In [ ]:
clean_df['ExpandingMean'] = (

    clean_df.groupby('Daerah')['IHK']

    .expanding()

    .mean()

    .reset_index(level=0, drop=True)
)

In [ ]:
clean_df.isnull().sum()

In [ ]:
model_df = clean_df.dropna()

In [ ]:
model_df.head()

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(
    model_df.drop(columns=['Daerah']).corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Korelasi Antar Feature")

plt.show()

In [ ]:
train_df = model_df[
    model_df['Tahun'] < 2025
]

test_df = model_df[
    model_df['Tahun'] >= 2025
]

In [ ]:
X_train = train_df.drop(
    columns=['Daerah', 'IHK']
)

y_train = train_df['IHK']

X_test = test_df.drop(
    columns=['Daerah', 'IHK']
)

y_test = test_df['IHK']

In [ ]:
model_df.to_csv(
    "dataset_forecasting_final.csv",
    index=False
)